In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Energy_tracker").getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()

Saving energy_usage_cleaned.csv to energy_usage_cleaned.csv


In [17]:
energy_usage_df = spark.read.csv(
    "energy_usage_cleaned.csv",
    header=True,
    inferSchema=True
    )

In [18]:
energy_usage_df.printSchema()

root
 |-- log_id: integer (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- device_name: string (nullable = true)
 |-- room_id: integer (nullable = true)
 |-- room_name: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- recorded_at: timestamp (nullable = true)



In [19]:
energy_usage_df.show()

+------+---------+-----------------+-------+-----------+----------+-------------------+
|log_id|device_id|      device_name|room_id|  room_name|energy_kwh|        recorded_at|
+------+---------+-----------------+-------+-----------+----------+-------------------+
|     1|        1|      Ceiling Fan|      1|Living Room|     0.075|2026-10-11 08:00:00|
|     2|        1|      Ceiling Fan|      1|Living Room|     0.075|2026-10-11 20:00:00|
|     3|        2|           LED TV|      1|Living Room|      0.12|2026-10-11 19:00:00|
|     4|        3|Living Room Light|      1|Living Room|     0.015|2026-10-11 18:00:00|
|     5|        5|     Refrigerator|      2|    Kitchen|      0.15|2026-10-11 06:00:00|
|     6|        5|     Refrigerator|      2|    Kitchen|     0.145|2026-10-11 18:00:00|
|     7|        6|   Microwave Oven|      2|    Kitchen|       0.3|2026-10-11 07:30:00|
|     8|        9|  Air Conditioner|      3|    Bedroom|       1.2|2026-10-11 22:00:00|
|     9|       10|      Bedroom 

In [20]:
#Here peak hours are between 18:00–21:59 and others are off-peak hours
from pyspark.sql.functions import when, col, hour, sum
energy_usage_df = energy_usage_df.withColumn(
    'hour',
    hour(col('recorded_at'))
)
energy_usage_df = energy_usage_df.withColumn(
    "usage_type",
    when((col("hour") >= 18) & (col("hour") < 22), "peak").otherwise("off_peak")
)
#group by device and calculate peak vs off-peak usage
energy_usage_df = energy_usage_df.fillna({
    'energy_kwh': 0.0
})
peak_offpeak_usage = energy_usage_df.groupBy('device_id').agg(
    sum(when(col('usage_type') == 'peak', col('energy_kwh')).otherwise(0)).alias('peak_usage'),
    sum(when(col('usage_type') == 'off_peak', col('energy_kwh')).otherwise(0)).alias('off_peak_usage')
).orderBy('device_id')
peak_offpeak_usage.show()

+---------+----------+-------------------+
|device_id|peak_usage|     off_peak_usage|
+---------+----------+-------------------+
|        1|     0.075|0.21000000000000002|
|        2|      0.48|                0.0|
|        3|      0.03|                0.0|
|        5|     0.293|0.45199999999999996|
|        6|       0.0|                0.6|
|        9|       0.0|               4.65|
|       10|      0.18|                0.0|
|       12|      0.02|                0.0|
|       13|       0.0|               0.98|
|       14|     0.009|                0.0|
|       16|      2.03|                0.0|
|       17|       0.0|                1.8|
+---------+----------+-------------------+



In [21]:
#top energy consuming devices
energy_usage_df.groupBy('device_id','device_name').agg(
    sum('energy_kwh').alias('total_energy_kwh')
).orderBy('total_energy_kwh', ascending=False).show()

+---------+-----------------+----------------+
|device_id|      device_name|total_energy_kwh|
+---------+-----------------+----------------+
|        9|  Air Conditioner|            4.65|
|       16|   Bathroom Light|            2.03|
|       17|     Water Heater|             1.8|
|       13| Desktop Computer|            0.98|
|        5|     Refrigerator|           0.745|
|        6|   Microwave Oven|             0.6|
|        2|           LED TV|            0.48|
|        1|      Ceiling Fan|           0.285|
|       10|      Bedroom Fan|            0.18|
|        3|Living Room Light|            0.03|
|       12|    Phone Charger|            0.02|
|       14|        Desk Lamp|           0.009|
+---------+-----------------+----------------+

